# 2 - Signature, enrichment and figures

From the nine per-dataset tables of notebook 1 to the consensus signature (Sections
4.3-4.4), its over-representation analysis (Section 4.5), the three figures and the
sensitivity analyses of Table S4.

| Output | Use |
|---|---|
| `results/Table2_signature.csv` | Table 2: signature genes, with direction and statistics |
| `results/TableS1_enrichment.csv` | Table S1: enriched terms |
| `results/Table3_pathways.csv` | Table 3: KEGG and Reactome pathways with FDR < 1e-4 |
| `figures/Figure1_dataset_selection.png`, `figures/Figure2_interventions.png`, `figures/Figure3_workflow.png` | Figures 1-3 |
| `results/TableS4_sensitivity.csv`, `results/TableS4_sensitivity_genes.csv` | Table S4: sensitivity analyses, as a summary and gene by gene |
| `results/orthologs_per_dataset.csv`, `results/signature_one_to_one_sensitivity.csv` | one-to-one orthologs: counts per dataset and gene-level results |

In [1]:
# Move to the project root (the folder that contains run_all.ps1), so that relative
# paths work whether the notebook is started from notebook/ or from the root.
root <- normalizePath(getwd(), winslash = "/", mustWork = TRUE)
while (!file.exists(file.path(root, "run_all.ps1")) && dirname(root) != root) {
  root <- dirname(root)
}
if (!file.exists(file.path(root, "run_all.ps1"))) {
  stop("project root not found: no run_all.ps1 above ", getwd())
}
setwd(root)
.libPaths(c(file.path(root, "env", "rlib"), .libPaths()))
Sys.setenv(RENAL_IRI_ROOT = root)
options(stringsAsFactors = FALSE, warn = 1)
cat(R.version.string, "\nroot: ", root, "\n", sep = "")

R version 4.4.3 (2025-02-28)
root: /Users/robertocasale/Desktop/lavoro_FBF/renal_shield_perfusion


## Setup

Shared settings (`R/helpers.R`), ortholog mapping and rank aggregation
(`R/pipeline_meta.R`), the signature rule (`R/pipeline_signature.R`) and the enrichment
(`R/pipeline_ora.R`).

In [2]:
source("R/helpers.R")
source("R/pipeline_meta.R")
source("R/pipeline_signature.R")
source("R/pipeline_ora.R")

[2026-10-08 17:26:45] INFO  helpers.R loaded; constants and robust_fetch() ready.



## Human orthologs

Every dataset is expressed in human gene symbols (Section 4.3).

In [3]:
fs <- list.files("results", "^DE_GSE.*[.]csv$", full.names = TRUE)
stopifnot(length(fs) == 9)
human_de <- list()
for (f in fs) {
  d <- read.csv(f)
  human_de[[unique(d$accession)[1]]] <- collapse_human(map_to_human(d, unique(d$organism)[1]))
}
cat("datasets:", paste(names(human_de), collapse = ", "), "\n")
cat("human genes per dataset:", paste(sapply(human_de, nrow), collapse = ", "), "\n")

[2026-10-08 17:27:05] INFO    Rattus norvegicus: 3293 outdated rodent symbols updated before ortholog mapping



[2026-10-08 17:27:09] INFO    Rattus norvegicus: 6106 outdated rodent symbols updated before ortholog mapping



[2026-10-08 17:27:22] INFO    Mus musculus: 1431 outdated rodent symbols updated before ortholog mapping



[2026-10-08 17:27:27] INFO    Mus musculus: 1763 outdated rodent symbols updated before ortholog mapping



[2026-10-08 17:27:32] INFO    Rattus norvegicus: 3917 outdated rodent symbols updated before ortholog mapping



[2026-10-08 17:27:36] INFO    Mus musculus: 1919 outdated rodent symbols updated before ortholog mapping



datasets: GSE126805, GSE148420, GSE27274, GSE30718, GSE34351, GSE39548, GSE43974, GSE58438, GSE98622 


human genes per dataset: 16854, 15216, 12862, 22839, 16123, 12784, 34212, 15165, 16852 


## Signature

Rank aggregation in each direction across the nine datasets, then the signature rule
(Section 4.4): genes measured in at least 5 of the 9 datasets, Benjamini-Hochberg adjusted
p < 0.05 in the assigned direction, and the same direction in at least half of the
datasets that measure the gene.

In [4]:
cons <- build_consensus(human_de)
sig  <- posthoc_signature(cons)
save_csv(sig[sig$signature, ], "Table2_signature.csv")
cat("consensus signature:", sum(sig$signature), "genes\n")
print(head(sig[sig$signature, c("human","mean_logFC","FDR_posthoc","concordant","n_datasets")], 12),
      row.names = FALSE, digits = 3)

[2026-10-08 17:27:48] INFO  Post-hoc signature: 256 genes (16540 tested at coverage >= 5)



[2026-10-08 17:27:48] INFO  WROTE       results/Table2_signature.csv (256 rows)



consensus signature: 256 genes


     human mean_logFC FDR_posthoc concordant n_datasets
   ADAMTS1       2.34    5.15e-09          9          9
       MYC       2.05    1.15e-07          9          9
      KLF6       2.33    5.07e-07          9          9
 TNFRSF12A       2.28    5.07e-07          9          9
     CLDN4       1.93    5.07e-07          9          9
     TGIF1       1.54    5.07e-07          9          9
     GDF15       2.25    1.25e-06          9          9
    NFKBIZ       1.93    1.25e-06          9          9
  SERPINE1       2.26    1.25e-06          9          9
      ELF3       1.26    1.94e-06          9          9
     ICAM1       1.37    2.07e-06          9          9
     SOCS3       2.12    2.07e-06          9          9


## Enrichment

WebGestalt over-representation analysis of the signature against the tested genes
(Section 4.5).

In [5]:
# The background is the set of tested genes (measured in at least 5 of the 9 datasets).
universe <- sig$human
cat("interest:", sum(sig$signature), "genes | background:", length(universe), "tested genes\n")
ora <- run_ora_set(sig$human[sig$signature], "signature", background = universe)
stopifnot(nrow(ora) > 0)
save_csv(ora, "TableS1_enrichment.csv")
cat(nrow(ora), "enriched terms\n")
print(table(ora$database))
# Table 3: KEGG and Reactome pathways with FDR < 1e-4.
table3 <- ora[grepl("^pathway", ora$database) & ora$FDR < 1e-4,
              c("database", "geneSet", "description", "size", "overlap", "enrichmentRatio", "FDR")]
save_csv(table3, "Table3_pathways.csv")
print(table3[, c("database", "description", "overlap", "enrichmentRatio", "FDR")], row.names = FALSE, digits = 3)

interest: 256 genes | background: 16540 tested genes


[2026-10-08 17:27:49] INFO  CACHE HIT   webgestalt_ora_signature_pathway_KEGG_customref_617c9d60_59db955d_n5_2000_f0.05 (queried 2026-09-13 18:49:37)



[2026-10-08 17:27:49] INFO  CACHE HIT   webgestalt_ora_signature_pathway_Reactome_customref_617c9d60_59db955d_n5_2000_f0.05 (queried 2026-09-13 18:49:44)



[2026-10-08 17:27:49] INFO  CACHE HIT   webgestalt_ora_signature_drug_DrugBank_customref_617c9d60_59db955d_n5_2000_f0.05 (queried 2026-09-13 18:49:50)



[2026-10-08 17:27:49] INFO  CACHE HIT   webgestalt_ora_signature_drug_GLAD4U_customref_617c9d60_59db955d_n5_2000_f0.05 (queried 2026-09-13 18:49:58)



[2026-10-08 17:27:49] INFO  WROTE       results/TableS1_enrichment.csv (134 rows)



134 enriched terms



   drug_DrugBank      drug_GLAD4U     pathway_KEGG pathway_Reactome 
               1               59               26               48 


[2026-10-08 17:27:49] INFO  WROTE       results/Table3_pathways.csv (13 rows)



         database                                               description
     pathway_KEGG                                     TNF signaling pathway
     pathway_KEGG                                   IL-17 signaling pathway
 pathway_Reactome                       Cytokine Signaling in Immune system
 pathway_Reactome                Interleukin-4 and Interleukin-13 signaling
 pathway_Reactome                                 Signaling by Interleukins
 pathway_Reactome                                  Interleukin-10 signaling
     pathway_KEGG                                    MAPK signaling pathway
 pathway_Reactome                                             Immune System
     pathway_KEGG                                                 Apoptosis
     pathway_KEGG                              NF-kappa B signaling pathway
     pathway_KEGG                                Osteoclast differentiation
 pathway_Reactome HSP90 chaperone cycle for steroid hormone receptors (SHR)
 pathway_Rea

## Figures

Figure 1 (dataset selection), Figure 2 (pathways and candidate interventions; the code
stops if a pathway or a gene drawn as part of the signature is no longer supported by the
results) and Figure 3 (workflow).

In [6]:
source("R/figures.R")
dir.create("figures", showWarnings = FALSE)
ident <- read.csv("results/geo_identification.csv", stringsAsFactors = FALSE)
cover <- if (file.exists("results/geo_corpus_coverage.csv"))
  read.csv("results/geo_corpus_coverage.csv", stringsAsFactors = FALSE) else NULL
invisible(fig_dataset_prisma(
  ident$n_identified[1], length(human_de), "figures/Figure1_dataset_selection.png",
  organisms = vapply(human_de, function(d) d$organism[1], character(1)),
  searched_on = ident$searched_on[1], coverage = cover, query = ident$query[1]))
common_name <- c("Homo sapiens" = "human", "Mus musculus" = "mouse", "Rattus norvegicus" = "rat")
orgs <- vapply(human_de, function(d) d$organism[1], character(1))
tb   <- table(ifelse(orgs %in% names(common_name), common_name[orgs], orgs))
invisible(fig_interventions("figures/Figure2_interventions.png", ora = ora, sig = sig))
invisible(fig_workflow("figures/Figure3_workflow.png",
  n_sig   = sum(sig$signature),
  n_terms = nrow(ora),
  n_path  = sum(grepl('^pathway', ora$database)),
  n_drug  = sum(grepl('^drug', ora$database)),
  n_datasets  = length(human_de),
  composition = paste(sprintf("%d %s", as.integer(tb), names(tb)), collapse = "   |   ") ))
cat("wrote figures/Figure1_dataset_selection.png, figures/Figure2_interventions.png and figures/Figure3_workflow.png
")

wrote figures/Figure1_dataset_selection.png, figures/Figure2_interventions.png and figures/Figure3_workflow.png


## Sensitivity analyses (Table S4)

The signature is derived again three times. Each run changes one step of the main
analysis, keeps every other step unchanged, and is compared with the 256 genes above:

1. one-to-one orthologs only (Section 4.3);
2. one Benjamini-Hochberg correction for both directions (Section 4.4);
3. GSE43974 with the two biopsies of the same kidney modelled as paired (Section 4.1).

This section runs after the figures because it loads the GEO functions of notebook 1.

### 1. One-to-one orthologs only

A rodent gene enters the analysis through its human ortholog. Most genes have exactly one.
Some belong to gene families that expanded differently in rodents and humans (for example
the CXCL chemokines); for these the main analysis uses the match supported by the most
orthology databases. This section counts the orthologs in each rodent dataset, then
removes every gene without a one-to-one ortholog and derives the signature again with the
same functions as above. The three human datasets are not changed.

A rodent-human pair is one-to-one when, among all the pairs that babelgene accepts (those
supported by at least three orthology databases), the rodent gene matches only that human
gene and the human gene matches only that rodent gene.

In [7]:
# 1. The one-to-one orthologs of each rodent species, from every pair that babelgene
#    accepts. top = FALSE returns all the matches of a gene, not only the best one, so a
#    gene with more than one match can be recognised.
species <- c("Mus musculus" = "mouse", "Rattus norvegicus" = "rat")
human_ids <- AnnotationDbi::keys(org.Hs.eg.db::org.Hs.eg.db, keytype = "ENTREZID")
one_to_one <- list()
for (org in names(species)) {
  p <- babelgene::orthologs(genes = human_ids, species = species[[org]],
                            human = TRUE, top = FALSE)
  p$human <- harmonise_human_symbol(p$human_symbol)   # the gene symbols the analysis uses
  n_rodent <- tapply(p$symbol, p$human, function(z) length(unique(z)))
  n_human  <- tapply(p$human, p$symbol, function(z) length(unique(z)))
  one_to_one[[org]] <- unique(p$human[n_rodent[p$human] == 1 & n_human[p$symbol] == 1])
}
# two genes with a known answer: MYC has a single ortholog in mouse and in rat, while
# CXCL2 belongs to the CXCL1/CXCL2/CXCL3 family, which has none
stopifnot(all(sapply(one_to_one, function(g) "MYC" %in% g)),
          !any(sapply(one_to_one, function(g) "CXCL2" %in% g)))

# 2. For each rodent dataset: the rodent genes measured, how many have a human ortholog
#    (the others are excluded by the mapping above), and how many of the resulting human
#    genes have a one-to-one ortholog.
orthologs_per_dataset <- do.call(rbind, lapply(fs, function(f) {
  d <- read.csv(f, stringsAsFactors = FALSE)
  org <- unique(d$organism)[1]
  if (!org %in% names(species)) return(NULL)
  acc <- unique(d$accession)[1]
  m <- suppressMessages(map_to_human(d, org))        # the same mapping as above
  m <- m[!is.na(m$gene) & !is.na(m$human) & m$human != "", ]
  rodent <- unique(d$gene[!is.na(d$gene) & d$gene != ""])
  data.frame(dataset = acc, species = species[[org]],
             rodent_genes = length(rodent),
             with_ortholog = length(unique(m$gene)),
             pct = round(100 * length(unique(m$gene)) / length(rodent), 1),
             human_genes = nrow(human_de[[acc]]),
             one_to_one = sum(human_de[[acc]]$human %in% one_to_one[[org]]))
}))
save_csv(orthologs_per_dataset, "orthologs_per_dataset.csv")
print(orthologs_per_dataset, row.names = FALSE)

[2026-10-08 17:28:15] INFO  WROTE       results/orthologs_per_dataset.csv (6 rows)



   dataset species rodent_genes with_ortholog  pct human_genes one_to_one
 GSE148420     rat        19297         15648 81.1       15216      14208
  GSE27274     rat        21791         13353 61.3       12862      12022
  GSE34351   mouse        21495         16458 76.6       16123      15096
  GSE39548   mouse        14330         12978 90.6       12784      11968
  GSE58438     rat        18784         15570 82.9       15165      14167
  GSE98622   mouse        29325         17451 59.5       16852      15661


In [8]:
# 3. The signature derived again from one-to-one orthologs only. Each rodent dataset keeps
#    only its one-to-one genes; the human datasets are unchanged. The derivation is the one
#    used above (build_consensus, posthoc_signature): only the input differs.
human_de_1to1 <- human_de
for (acc in orthologs_per_dataset$dataset) {
  org <- names(species)[species == orthologs_per_dataset$species[orthologs_per_dataset$dataset == acc]]
  human_de_1to1[[acc]] <- human_de[[acc]][human_de[[acc]]$human %in% one_to_one[[org]], ]
}
sig_1to1 <- posthoc_signature(build_consensus(human_de_1to1))

# A gene is tested only if it is measured in at least 5 of the 9 datasets. Some signature
# genes keep fewer once their rodent values are removed, so this run cannot test them;
# they are counted apart from the genes that are tested and not found again.
main     <- sig$human[sig$signature]
alt      <- sig_1to1$human[sig_1to1$signature]
testable <- main[main %in% sig_1to1$human]
found    <- intersect(testable, alt)
same_direction <- all(sig$direction[match(found, sig$human)] ==
                      sig_1to1$direction[match(found, sig_1to1$human)])
cat(sprintf("signature genes of the main analysis: %d\n", length(main)))
cat(sprintf("  not testable without rodent values (fewer than 5 datasets): %d\n    %s\n",
            length(setdiff(main, testable)), paste(sort(setdiff(main, testable)), collapse = ", ")))
cat(sprintf("  testable: %d, of which found again: %d (%.1f%%), same direction: %s\n",
            length(testable), length(found), 100 * length(found) / length(testable), same_direction))
cat(sprintf("  testable and not found again: %d\n    %s\n",
            length(setdiff(testable, alt)), paste(sort(setdiff(testable, alt)), collapse = ", ")))
cat(sprintf("new genes in the one-to-one signature: %d\n    %s\n",
            length(setdiff(alt, main)), paste(sort(setdiff(alt, main)), collapse = ", ")))
both <- union(main, alt)
save_csv(data.frame(human = both,
                    main_signature = both %in% main,
                    one_to_one_signature = both %in% alt,
                    testable_one_to_one = both %in% sig_1to1$human,
                    FDR_main = sig$FDR_posthoc[match(both, sig$human)],
                    FDR_one_to_one = sig_1to1$FDR_posthoc[match(both, sig_1to1$human)]),
         "signature_one_to_one_sensitivity.csv")

[2026-10-08 17:28:22] INFO  Post-hoc signature: 234 genes (15675 tested at coverage >= 5)



signature genes of the main analysis: 256


  not testable without rodent values (fewer than 5 datasets): 14
    AKR1C2, ANXA2, BTG1, CCL2, CXCL2, CYP4A22, HSPA1A, MT1F, MT2A, PLSCR1, SERPINA3, TUBA1C, TUBB2A, TUBB2B


  testable: 242, of which found again: 232 (95.9%), same direction: TRUE


  testable and not found again: 10
    CLIC1, HMGA1, MYL12A, S100A11, SFN, SLC19A2, SLC25A25, SLCO4C1, SRXN1, TBRG1


new genes in the one-to-one signature: 2
    MYH9, TAX1BP3


[2026-10-08 17:28:22] INFO  WROTE       results/signature_one_to_one_sensitivity.csv (258 rows)



### 2. One correction for both directions

In the main analysis each tested gene is assigned the direction with the smaller
aggregation score, and the Benjamini-Hochberg correction is applied separately to the
genes assigned up and to those assigned down. Here both scores of every tested gene,
2 x 16,540 = 33,080 p-values, are corrected together. The direction of each gene and the
agreement rule are unchanged.

In [9]:
# Both aggregation scores of every tested gene (up and down) corrected together, then the
# same direction and the same agreement rule as in the main analysis.
n_tested  <- nrow(sig)
fdr_joint <- p.adjust(c(sig$RRA_up, sig$RRA_down), method = "BH")
stopifnot(length(fdr_joint) == 2 * n_tested, !anyNA(fdr_joint))
sig_joint <- sig
sig_joint$FDR_up_joint   <- fdr_joint[seq_len(n_tested)]
sig_joint$FDR_down_joint <- fdr_joint[n_tested + seq_len(n_tested)]
sig_joint$FDR_posthoc <- ifelse(sig_joint$direction == "up",
                                sig_joint$FDR_up_joint, sig_joint$FDR_down_joint)
sig_joint$signature <- sig_joint$FDR_posthoc < SIG_RULE$FDR_MAX &
                       sig_joint$concordant >= pmax(2L, ceiling(sig_joint$n_datasets / 2))
# no gene may be below the threshold in both directions: it would need a rule to choose one
n_both <- sum(sig_joint$FDR_up_joint < SIG_RULE$FDR_MAX & sig_joint$FDR_down_joint < SIG_RULE$FDR_MAX)
stopifnot(n_both == 0)
alt_joint <- sig_joint$human[sig_joint$signature]
cat(sprintf("p-values corrected together: %d (%d tested genes x 2 directions)\n",
            length(fdr_joint), n_tested))
cat(sprintf("genes below the threshold in both directions: %d\n", n_both))
cat(sprintf("signature with the joint correction: %d genes (%d up, %d down)\n", length(alt_joint),
            sum(sig_joint$direction[sig_joint$signature] == "up"),
            sum(sig_joint$direction[sig_joint$signature] == "down")))
cat(sprintf("  signature genes of the main analysis found again: %d of %d\n",
            length(intersect(main, alt_joint)), length(main)))
cat(sprintf("  new genes: %d\n    %s\n", length(setdiff(alt_joint, main)),
            paste(sort(setdiff(alt_joint, main)), collapse = ", ")))

p-values corrected together: 33080 (16540 tested genes x 2 directions)


genes below the threshold in both directions: 0


signature with the joint correction: 175 genes (152 up, 23 down)


  signature genes of the main analysis found again: 171 of 256


  new genes: 4
    DNAJC28, LHX1, TNFRSF14, UGT8


### 3. GSE43974 with paired biopsies

GSE43974 compares biopsies taken at the end of cold ischemia (T2) with biopsies taken
after reperfusion (T3), and the main analysis treats its 332 biopsies as two independent
groups. For 96 deceased-donor kidneys both biopsies were profiled. The sample annotations
of GEO mark these biopsies as paired but do not say which T2 and T3 biopsies come from
the same kidney; the pairs are listed, by array barcode, in a supplementary file of the
series (`GSE43974_paired_analyses_DBD_DCD_Living.xls`). Here the same 332 biopsies are
fitted again with the kidney as a random block (limma `duplicateCorrelation`, as for
GSE27274 in notebook 1); a biopsy without its pair is a block of its own. The signature
is then derived again with this table in place of the main GSE43974 table. The block
model is the slowest step of the notebook: 40 minutes or more, depending on the computer.

In [10]:
# 1. The pairs. The supplementary file of GSE43974 lists, for each kidney, the array
#    barcodes of its T2 and T3 biopsies, with one sheet per donor type (DBD brain-dead,
#    DCD cardiac-dead, Living); the living donors are not analysed. The sheets are
#    downloaded once and cached like every other GEO answer.
source("R/pipeline_de.R")
sheets <- robust_fetch("geo_suppl_GSE43974_paired_analyses_sheets", fn = function() {
  f  <- rownames(GEOquery::getGEOSuppFiles("GSE43974", baseDir = DIRS$geo,
                                           filter_regex = "paired_analyses"))
  xl <- sub("[.]gz$", "", f)
  R.utils::gunzip(f, xl, remove = FALSE, overwrite = TRUE)
  lapply(setNames(nm = readxl::excel_sheets(xl)),
         function(s) as.data.frame(readxl::read_excel(xl, sheet = s)))
})
stopifnot("GSE43974 pairs file not available" = sheets$ok)
pairs <- do.call(rbind, lapply(c("DBD", "DCD"), function(s) {
  x <- sheets$value[[s]]
  data.frame(kidney = sprintf("%s_%02d", s, seq_len(nrow(x))), T2 = x$T2, T3 = x$T3)
}))
stopifnot(nrow(pairs) == 96)

# 2. The 332 biopsies of the main analysis (the loader of notebook 1), with their array
#    barcode (the "description" field of each GEO sample). A biopsy of a pair gets the
#    kidney as its block; every other biopsy is a block of its own.
gse43974 <- load_GSE43974()
pd <- Biobase::pData(get_geo_es("GSE43974"))
meta43974 <- gse43974$meta
meta43974$barcode <- as.character(pd$description[match(meta43974$sample, rownames(pd))])
i2 <- match(pairs$T2, meta43974$barcode)
i3 <- match(pairs$T3, meta43974$barcode)
meta43974$kidney <- meta43974$sample
meta43974$kidney[i2] <- pairs$kidney
meta43974$kidney[i3] <- pairs$kidney
in_pair <- !is.na(pd[["deceased paired biopsies:ch1"]][match(meta43974$sample, rownames(pd))])
stopifnot(nrow(meta43974) == 332, !anyNA(meta43974$barcode),
          !anyDuplicated(c(pairs$T2, pairs$T3)),
          !anyNA(c(i2, i3)),                                   # every paired biopsy is analysed
          all(meta43974$group[i2] == "control"),               # T2: end of cold ischemia
          all(meta43974$group[i3] == "IRI"),                   # T3: after reperfusion
          identical(in_pair, meta43974$kidney != meta43974$sample))  # GEO flags the same biopsies
cat(sprintf("GSE43974: %d biopsies, %d kidneys with both biopsies (%d DBD, %d DCD), %d unpaired biopsies\n",
            nrow(meta43974), nrow(pairs), sum(grepl("^DBD", pairs$kidney)),
            sum(grepl("^DCD", pairs$kidney)), sum(!in_pair)))

[2026-10-08 17:28:28] INFO  CACHE HIT   geo_suppl_GSE43974_paired_analyses_sheets (queried 2026-10-08 17:25:38)



[2026-10-08 17:28:32] INFO  CACHE HIT   geo_probe_GSE43974 (queried 2026-09-11 18:43:27)



[2026-10-08 17:28:40] INFO  GSE43974: 163 control (T2, post-cold-ischemia) vs 169 IRI (T3, post-reperfusion), unpaired, full cohort



[2026-10-08 17:28:43] INFO  CACHE HIT   geo_probe_GSE43974 (queried 2026-09-11 18:43:27)



GSE43974: 332 biopsies, 96 kidneys with both biopsies (67 DBD, 29 DCD), 140 unpaired biopsies


In [11]:
# 3. limma on the same expression values and genes as notebook 1 (run_de, microarray
#    branch), with the kidney as a random block. The same code without the block must
#    reproduce results/DE_GSE43974.csv, so that the block is the only difference.
mat43974 <- gse43974$mat
mat43974 <- mat43974[rowSums(!is.na(mat43974)) >= ncol(mat43974) * 0.7, , drop = FALSE]
mat43974 <- mat43974[rowSums(mat43974 != 0, na.rm = TRUE) > 0, , drop = FALSE]
design43974 <- model.matrix(~ factor(meta43974$group, levels = c("control", "IRI")))
de_table <- function(fit) {
  res <- topTable(eBayes(fit), coef = 2, number = Inf, sort.by = "none")
  out <- data.frame(gene = rownames(res), logFC = res$logFC, AveExpr = res$AveExpr,
                    P.Value = res$P.Value, adj.P.Val = res$adj.P.Val)
  out <- out[!is.na(out$P.Value) & !is.na(out$logFC), ]
  out[order(out$P.Value), ]
}
unblocked <- de_table(lmFit(mat43974, design43974))
reported  <- read.csv("results/DE_GSE43974.csv")
stopifnot(identical(unblocked$gene, reported$gene),
          isTRUE(all.equal(unblocked$logFC, reported$logFC, tolerance = 1e-10)),
          isTRUE(all.equal(unblocked$P.Value, reported$P.Value, tolerance = 1e-10)))

dc43974 <- duplicateCorrelation(mat43974, design43974, block = meta43974$kidney)
paired43974 <- de_table(lmFit(mat43974, design43974, block = meta43974$kidney,
                              correlation = dc43974$consensus))
cat(sprintf("within-kidney correlation (consensus): %.3f\n", dc43974$consensus))
m <- match(reported$gene, paired43974$gene)
cat(sprintf("paired vs main GSE43974 table: %d genes; Spearman correlation of log2 fold changes %.3f, of p-values %.3f\n",
            nrow(paired43974), cor(reported$logFC, paired43974$logFC[m], method = "spearman"),
            cor(reported$P.Value, paired43974$P.Value[m], method = "spearman")))

# 4. The signature derived again with this table in place of the main GSE43974 table;
#    the other eight datasets are unchanged.
paired43974$accession <- "GSE43974"; paired43974$organism <- "Homo sapiens"
human_de_paired <- human_de
human_de_paired[["GSE43974"]] <- collapse_human(map_to_human(paired43974, "Homo sapiens"))
sig_paired <- posthoc_signature(build_consensus(human_de_paired))
alt_paired <- sig_paired$human[sig_paired$signature]
cat(sprintf("signature with GSE43974 paired: %d genes (%d up, %d down)\n", length(alt_paired),
            sum(sig_paired$direction[sig_paired$signature] == "up"),
            sum(sig_paired$direction[sig_paired$signature] == "down")))
cat(sprintf("  signature genes of the main analysis found again: %d of %d\n",
            length(intersect(main, alt_paired)), length(main)))
cat(sprintf("  new genes: %d\n    %s\n", length(setdiff(alt_paired, main)),
            paste(sort(setdiff(alt_paired, main)), collapse = ", ")))

within-kidney correlation (consensus): 0.161


paired vs main GSE43974 table: 34694 genes; Spearman correlation of log2 fold changes 0.999, of p-values 0.998


[2026-10-08 18:17:31] INFO  Post-hoc signature: 257 genes (16540 tested at coverage >= 5)



signature with GSE43974 paired: 257 genes (238 up, 19 down)


  signature genes of the main analysis found again: 256 of 256


  new genes: 1
    MORF4L2


### Table S4

For each run: the genes tested (measured in at least 5 of the 9 datasets), the signature
genes, how many of the 256 genes of the main analysis are found again, cannot be tested or
are not found again, and the new genes. The code stops if a gene found again has changed
direction.

In [12]:
# Table S4: the main analysis and the three sensitivity analyses, compared with the 256
# genes of the main signature. A gene found again must have the same direction.
compare_with_main <- function(analysis, s) {
  alt   <- s$human[s$signature]
  found <- intersect(main, alt)
  stopifnot(all(sig$direction[match(found, sig$human)] == s$direction[match(found, s$human)]))
  data.frame(analysis = analysis,
             genes_tested = nrow(s),
             signature_genes = length(alt),
             up = sum(s$direction[s$signature] == "up"),
             down = sum(s$direction[s$signature] == "down"),
             main_genes_found = length(found),
             main_genes_not_testable = sum(!main %in% s$human),
             main_genes_not_found = sum(main %in% s$human & !main %in% alt),
             new_genes = length(setdiff(alt, main)),
             new_gene_symbols = paste(sort(setdiff(alt, main)), collapse = "; "))
}
runs <- list(main = sig, one_to_one = sig_1to1, joint = sig_joint, gse43974_paired = sig_paired)
labels <- c(main = "Main analysis", one_to_one = "One-to-one orthologs only",
            joint = "One correction for both directions",
            gse43974_paired = "GSE43974 with paired biopsies")
s4 <- do.call(rbind, Map(compare_with_main, labels[names(runs)], runs))
rownames(s4) <- NULL
stopifnot(s4$main_genes_found + s4$main_genes_not_testable + s4$main_genes_not_found == length(main),
          s4$up + s4$down == s4$signature_genes)
save_csv(s4, "TableS4_sensitivity.csv")
print(s4[, names(s4) != "new_gene_symbols"], row.names = FALSE)

# The same comparison gene by gene: every gene in at least one of the four signatures and,
# for each run, whether it is in the signature (NA when that run cannot test it), its
# assigned direction and its adjusted p-value in that direction.
genes <- unique(unlist(lapply(runs, function(s) s$human[s$signature])))
s4_genes <- data.frame(human = genes)
for (r in names(runs)) {
  i <- match(genes, runs[[r]]$human)
  s4_genes[[paste0("signature_", r)]] <- runs[[r]]$signature[i]
  s4_genes[[paste0("direction_", r)]] <- runs[[r]]$direction[i]
  s4_genes[[paste0("FDR_", r)]]       <- runs[[r]]$FDR_posthoc[i]
}
save_csv(s4_genes, "TableS4_sensitivity_genes.csv")

[2026-10-08 18:17:31] INFO  WROTE       results/TableS4_sensitivity.csv (4 rows)



                           analysis genes_tested signature_genes  up down
                      Main analysis        16540             256 237   19
          One-to-one orthologs only        15675             234 218   16
 One correction for both directions        16540             175 152   23
      GSE43974 with paired biopsies        16540             257 238   19
 main_genes_found main_genes_not_testable main_genes_not_found new_genes
              256                       0                    0         0
              232                      14                   10         2
              171                       0                   85         4
              256                       0                    0         1


[2026-10-08 18:17:31] INFO  WROTE       results/TableS4_sensitivity_genes.csv (263 rows)



Next: `03_validation.ipynb`.